# Importando Bibliotecas

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from scipy.stats import zscore
import pingouin as pg
from os import replace
import warnings
warnings.filterwarnings("ignore")

# Lendo base de dados anonimizada

In [2]:
df = pd.read_excel('base_dados_anonimizada.xlsx')
df_copia = df.copy()

# Tratamento da base de dados anonimizada

## Tratamento de nulos e exclusão de variáveis irrelevantes

### Excluindo colunas

In [3]:
# 1. Definimos a lista exata de colunas que queremos remover
colunas_para_excluir = ["REVENDA", "SITUACAO", "DESLOCAMENTO", "SERVICO GUINCHO"]

# 2. Exclusão das colunas
df = df.drop(columns=colunas_para_excluir, errors="ignore")

# 3. Visualização do resultado
print(df.columns.tolist())

['ID_OS', 'ID_Cliente', 'ENTRADA VEICULO', 'VEICULO PRONTO', 'SAIDA VEICULO', 'VALOR TOTAL', 'NOME VEICULO', 'DEFEITO', 'MARCA', 'ID_PLACA', 'QUILOMETRAGEM', 'DATA GARANTIA', 'VALOR ADIANTAMENTO', 'VALOR SERVICO', 'VALOR PECAS', 'OUTROS VALORES']


### Variavel DEFEITOS

In [4]:
# 1. Verifica a quantidade de preenchimento das células
qtd_preenchidos_defeitos = df["DEFEITO"].notna().sum()
total_linhas = len(df)
porcentagem_preenchidos_defeitos = (qtd_preenchidos_defeitos / total_linhas) * 100

# Visualização
print(f"Registros com a coluna DEFEITO preenchida: {qtd_preenchidos_defeitos}")
print(f"Isso representa {porcentagem_preenchidos_defeitos:.2f}% da base de dados.\n")

# 2. Preencher os valores vazios (NaN) com "NÃO INFORMADO"
df["DEFEITO"] = df["DEFEITO"].fillna("NÃO INFORMADO")

Registros com a coluna DEFEITO preenchida: 211
Isso representa 1.81% da base de dados.



### Variáveis derivadas

In [5]:
# 1. Preparação: Garantir formato numérico e de data para TODOS os campos de data
df["ENTRADA VEICULO"] = pd.to_datetime(df["ENTRADA VEICULO"], format="%d/%m/%Y %H:%M:%S", errors="coerce")
df["SAIDA VEICULO"] = pd.to_datetime(df["SAIDA VEICULO"], format="%d/%m/%Y %H:%M:%S", errors="coerce")
df["VEICULO PRONTO"] = pd.to_datetime(df["VEICULO PRONTO"], format="%d/%m/%Y %H:%M:%S", errors="coerce")

df["VALOR PECAS"] = pd.to_numeric(df["VALOR PECAS"], errors="coerce").fillna(0)

# 2. TEMPO_PERMANENCIA_H
# Calcula a diferença e converte o tempo total para horas decimais
df["TEMPO_PERMANENCIA_H"] = (df["SAIDA VEICULO"] - df["ENTRADA VEICULO"]).dt.total_seconds() / 3600

# 3. TEMPO_EXECUCAO_H
# Calcula a diferença entre a finalização do serviço e a entrada
df["TEMPO_EXECUCAO_H"] = (df["VEICULO PRONTO"] - df["ENTRADA VEICULO"]).dt.total_seconds() / 3600

# 4. TEM_PECAS
df["TEM_PECAS"] = np.where(df["VALOR PECAS"] > 0, 1, 0)

# 5. ANO e MES
df["ANO"] = df["ENTRADA VEICULO"].dt.year
df["MES"] = df["ENTRADA VEICULO"].dt.month

# 6. MARCA_NORM
df["MARCA_NORM"] = df["MARCA"]

## Tratamento de anomalias financeiras e de quilometragem

### Ordens de serviço com valor total igual a zero

In [6]:
# 1. Garantir que a coluna "VALOR TOTAL" seja do tipo numérico
df["VALOR TOTAL"] = pd.to_numeric(df["VALOR TOTAL"], errors="coerce") # Transforma "não numeros" em NaT

# 2. Criar um filtro onde o valor é zero
filtro_valor_zero = df["VALOR TOTAL"] == 0

# 3. Cópia do dataframe apenas com as OS zeradas
df_zeradas = df[filtro_valor_zero]

# 6. Visualização
quantidade_zeradas = len(df_zeradas)
quantidade_linhas = len(df)
porcentagem_zeradas = (quantidade_zeradas / quantidade_linhas) * 100

print(f"Foram encontradas {quantidade_zeradas} Ordens de Serviço com Valor Total igual a zero.")
print(f"Sendo {porcentagem_zeradas:.2f}% do total de Ordens de Serviço.")

# Visualização do dataframe (apenas os primeiros 25)
# df_zeradas.head(25)

Foram encontradas 68 Ordens de Serviço com Valor Total igual a zero.
Sendo 0.58% do total de Ordens de Serviço.


### Valores negativos em OUTROS VALORES

In [7]:
# 1. Garantir que a coluna "OUTROS VALORES" seja do tipo numérico
df["OUTROS VALORES"] = pd.to_numeric(df["OUTROS VALORES"], errors="coerce")

# 2. Criar filtro para onde o valor é menor que zero
filtro_negativo = df["OUTROS VALORES"] < 0

# 3. Cópia do dataframe apenas com os valores negativos
df_negativos = df[filtro_negativo]

# 4. Visualização
quantidade_negativos = len(df_negativos)
total_descontos = df_negativos["OUTROS VALORES"].sum()

print(f'Foram encontradas {quantidade_negativos} Ordens de Serviço com "OUTROS VALORES" negativos.')
print(f"Sendo um total de R$ {total_descontos:.2f}")

# Visualisação da cópia do dataframe (Apenas 25 primeiros em ordem do mais negativo ao menos negativo)
# df_negativos.nsmallest(25, "OUTROS VALORES")

Foram encontradas 370 Ordens de Serviço com "OUTROS VALORES" negativos.
Sendo um total de R$ -43629.52


### Quilometragem ausente ou nula

In [8]:
# 1. Garantir que a coluna "QUILOMETRAGEM" seja tratada como número
df["QUILOMETRAGEM"] = pd.to_numeric(df["QUILOMETRAGEM"], errors="coerce")

# 2. Contar quantos zeros eistem e sua porcentagem do total
qtd_km_zero = len(df[df["QUILOMETRAGEM"] == 0])
total_linhas = len(df)
porcentagem_km_zero = (qtd_km_zero / total_linhas) * 100

# Visualização
print(f"Foram encontradas {qtd_km_zero} Ordens de Serviço com Quilometragem igual a zero.")
print(f"Sendo {porcentagem_km_zero:.2f}% do total de Ordens de Serviço.")

# 3. Substituir os zeros por nulo (NaN) no dataframe
df["QUILOMETRAGEM"] = df["QUILOMETRAGEM"].replace(0, np.nan)

Foram encontradas 2115 Ordens de Serviço com Quilometragem igual a zero.
Sendo 18.10% do total de Ordens de Serviço.


## Padronização e visualização da variável MARCA

In [9]:
# 1. Padronizando maiúsculas e removendo espaçõs em brancos "nas pontas"
df["MARCA"] = df["MARCA"].astype(str).str.upper().str.strip()

# 2. Removendo quebras de linha indesejadas
df["MARCA"] = df["MARCA"].str.replace("\n", "", regex=False)

# 3. Criando dicionário de correção de grafias
correcoes_marcas = {
    # Variações da Volkswagen
    "VW": "VOLKSWAGEN",
    "VOLKSWAGEM": "VOLKSWAGEN",
    "VOLKWAGEN": "VOLKSWAGEN",

    # Variações da Chevrolet
    "GM": "CHEVROLET",
    "CHEVROLETE": "CHEVROLET",

    # Variações da Renault
    "RENALT": "RENAULT",
    "SANDERO": "RENAULT", # Sandero é modelo

    # Variações da Mercedes-Benz
    "MERCEDES": "MERCEDES-BENZ",
    "MERCEDES BENS": "MERCEDES-BENZ",

    # Variações e Modelos de outras marcas
    "CAOA CHERY": "CHERY",
    "KIA MOTORS": "KIA",
    "MMC": "MITSUBISHI",
    "ESCORT": "FORD",       # Escort é modelo
    "AZERA": "HYUNDAI",     # Azera é modelo
    "IMPREZA": "SUBARU",    # Impreza é modelo
    "RANGE ROVER": "LAND ROVER",
    "DAEWOD": "DAEWOO",     # Erro de digitação comum

    # Tratando Placas que caíram na coluna errada (transformando em nulo/NaN)
    "BJA 4103": np.nan,
    "BKX 8258": np.nan,
    "MOP 2H11": np.nan,
    "NAN": np.nan           # Volta a marcação de vazio para o padrão do Pandas
}
# 4. Aplicamos a correção
df["MARCA"] = df["MARCA"].replace(correcoes_marcas)

# 5. Visualizar o resultado final
# A contagem foi separada em uma variável para manter o uso consistente das aspas duplas dentro do print
quantidade_marcas = df["MARCA"].nunique()
print(f"Número de marcas únicas agora: {quantidade_marcas}\n")
print(df["MARCA"].value_counts(dropna=False).to_string())

Número de marcas únicas agora: 44

MARCA
VOLKSWAGEN       2259
CHEVROLET        2174
FIAT             2015
FORD             1352
RENAULT           602
PEUGEOT           539
HYUNDAI           538
HONDA             537
CITROEN           310
TOYOTA            252
NISSAN            214
TROLLER           139
MITSUBISHI        126
NaN               105
JAC MOTORS         94
KIA                85
JEEP               74
AUDI               62
BMW                40
MERCEDES-BENZ      26
CHRYSLER           26
CHERY              22
ENGESA             21
DODGE              13
LAND ROVER         12
VOLVO              12
SUZUKI              9
PUMA                6
CAMPER              3
SUBARU              2
BRM                 2
AGRALE              2
ENVEMO              1
DAEWOO              1
ALFA ROMEO          1
PORSCHE             1
GURGEL              1
EFFA                1
MASERATI            1
LIFAN               1
MINI                1
SMART               1
JAGUAR              1
LEXUS        

## Conversão de datas e cálculo de tempo de permanência

In [10]:
# 1. Erros de conversão de data (o que virou NaT na conversão)
# Verifica se a coluna original tinha texto, mas a convertida virou NaT
erros_entrada = df["ENTRADA VEICULO"].notna() & pd.to_datetime(df["ENTRADA VEICULO"], format="%d/%m/%Y %H:%M:%S", errors="coerce").isna()
erros_saida = df["SAIDA VEICULO"].notna() & pd.to_datetime(df["SAIDA VEICULO"], format="%d/%m/%Y %H:%M:%S", errors="coerce").isna()

print(f"Erros de formato na Entrada: {erros_entrada.sum()}")
print(f"Erros de formato na Saída: {erros_saida.sum()}")

# Aplicando as conversões de fato
df["ENTRADA VEICULO"] = pd.to_datetime(df["ENTRADA VEICULO"], format="%d/%m/%Y %H:%M:%S", errors="coerce")
df["SAIDA VEICULO"] = pd.to_datetime(df["SAIDA VEICULO"], format="%d/%m/%Y %H:%M:%S", errors="coerce")

# 2. Criar a coluna de tempo de permanência
df["TEMPO_PERMANENCIA"] = df["SAIDA VEICULO"] - df["ENTRADA VEICULO"]

# 3. Contando tempos inválidos (Saída antes da Entrada)
filtro_invalido = df["TEMPO_PERMANENCIA"] < pd.Timedelta(days=0)
qtd_negativos = filtro_invalido.sum()
print(f"Registros com saída antes da entrada: {qtd_negativos}")

# 4. Contando tempos maiores que 90 dias
filtro_90_dias = df["TEMPO_PERMANENCIA"] > pd.Timedelta(days=90)
qtd_acima_90 = filtro_90_dias.sum()
print(f"Registros com permanência > 90 dias: {qtd_acima_90}")

Erros de formato na Entrada: 0
Erros de formato na Saída: 0
Registros com saída antes da entrada: 7
Registros com permanência > 90 dias: 110


In [11]:
# 1. Converter as colunas para formato de data
df["ENTRADA VEICULO"] = pd.to_datetime(df["ENTRADA VEICULO"], format="%d/%m/%Y %H:%M:%S", errors="coerce")
df["SAIDA VEICULO"] = pd.to_datetime(df["SAIDA VEICULO"], format="%d/%m/%Y %H:%M:%S", errors="coerce")

# 2. Criar uma nova coluna com o tempo de permanência
df["TEMPO_PERMANENCIA"] = df["SAIDA VEICULO"] - df["ENTRADA VEICULO"]

# 3. Filtrando tempos inválidos
filtro_invalido = df["TEMPO_PERMANENCIA"] < pd.Timedelta(days=0) # Saida registrada antes da entrada


# 4. Editando dados para NaT (Not a Time)
df.loc[filtro_invalido, ["ENTRADA VEICULO", "SAIDA VEICULO", "TEMPO_PERMANENCIA"]] = pd.NaT

# 5. Excluindo tempos maiores que 90 dias
filtro_90_dias = df["TEMPO_PERMANENCIA"] > pd.Timedelta(days=90) # Tempo de permanencia maiores que 90 dias
df = df[~filtro_90_dias] # Mantem todos os dados menos os do "filtro_90_dias"

# 6. Visualização do dataframe
# df